In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    auc,
)

df = pd.read_csv("..\\data\\online_shoppers_intention.csv")
df.head()


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [2]:
# ==============================
# Data Preprocessing
# ==============================

# 1. Boolean columns to integer
df[["Weekend", "Revenue"]] = df[["Weekend", "Revenue"]].astype(int)

# 2. One-hot encode VisitorType
df = pd.get_dummies(df, columns=['VisitorType'], drop_first=True)

# 3. Map month names to numbers
month_map = {'Feb': 2, 'Mar': 3, 'May': 5, 'Oct': 10, 'June': 6, 'Jul': 7,
            'Aug': 8, 'Nov': 11, 'Sep': 9, 'Dec': 12}
df["Month"] = df["Month"].map(month_map)

# 4. Feature engineering: total page views and duration
df["Total_PageViews"] = df["Administrative"] + df["Informational"] + df["ProductRelated"]
df["Total_PageDuration"] = df["Administrative_Duration"] + df["Informational_Duration"] + df["ProductRelated_Duration"]

# 5. Drop original columns
df.drop(columns=["Administrative", "Informational", "ProductRelated",
                "Administrative_Duration", "Informational_Duration", "ProductRelated_Duration"], inplace=True)

# Preview processed data
df.head()


,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,Weekend,Revenue,VisitorType_Other,VisitorType_Returning_Visitor,Total_PageViews,Total_PageDuration
0,0.20,0.20,0.0,0.0,2,1,1,1,1,0,0,False,True,1,0.000000
1,0.00,0.10,0.0,0.0,2,2,2,1,2,0,0,False,True,2,64.000000
2,0.20,0.20,0.0,0.0,2,4,1,9,3,0,0,False,True,1,0.000000
3,0.05,0.14,0.0,0.0,2,3,2,2,4,0,0,False,True,2,2.666667
4,0.02,0.05,0.0,0.0,2,3,3,1,4,1,0,False,True,10,627.500000


In [3]:
# Train-test split (stratified)
X = df.drop('Revenue', axis=1)
y = df['Revenue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.3, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
pos_weight = df["Revenue"].value_counts()[0] / df["Revenue"].value_counts()[1]
# Evaluate multiple models
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", max_iter=1000, class_weight="balanced", random_state=42),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "xgboost": XGBClassifier(use_label_encoder=False,
    eval_metric="logloss",
    scale_pos_weight=pos_weight,
    random_state=42,),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}
metrics = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = None
    
    metrics[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc
    }

pd.DataFrame(metrics).T


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22492\663696171.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pos_weight = df["Revenue"].value_counts()[0] / df["Revenue"].value_counts()[1]
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\xgboost\training.py:183: UserWarning: [10:13:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,Accuracy,Precision,Recall,F1,ROC-AUC
Logistic Regression,0.863206,0.543307,0.723776,0.620690,0.885649
SVM,0.883752,0.684896,0.459790,0.550209,0.874467
Decision Tree,0.847526,0.506623,0.534965,0.520408,0.719833
Random Forest,0.898081,0.738386,0.527972,0.615698,0.914385
xgboost,0.881049,0.603774,0.671329,0.635762,0.908116
KNN,0.871587,0.639769,0.388112,0.483134,0.799142


In [23]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score

# Create a pipeline: SMOTE + Random Forest
pipe = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(
        n_estimators=500,
        max_depth=10,
        min_samples_split=8,
        min_samples_leaf=4,
        max_features='log2',
        class_weight='balanced',
        random_state=42
    ))
])

pipe.fit(X_train_scaled, y_train)

# 5-fold stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get predicted probabilities (for threshold tuning)
y_pred_prob_train = cross_val_predict(pipe, X_train_scaled, y_train, 
                            cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
y_pred_prob_test = pipe.predict_proba(X_test_scaled)[:, 1]




In [24]:
# Threshold for class prediction
chosen_threshold = 0.7  # or test a few, as you did before
y_pred = (y_pred_prob_train >= chosen_threshold).astype(int)

# Evaluate
print("Confusion Matrix with SMOTE and threshold {:.2f}:".format(chosen_threshold))
print(confusion_matrix(y_train, y_pred))
print(classification_report(y_train, y_pred))
print("Precision:", precision_score(y_train, y_pred))
print("Recall:   ", recall_score(y_train, y_pred))
print("F1-Score: ", f1_score(y_train, y_pred))

Confusion Matrix with SMOTE and threshold 0.70:
[[6860  435]
 [ 446  890]]
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      7295
           1       0.67      0.67      0.67      1336

    accuracy                           0.90      8631
   macro avg       0.81      0.80      0.80      8631
weighted avg       0.90      0.90      0.90      8631

Precision: 0.6716981132075471
Recall:    0.6661676646706587
F1-Score:  0.6689214580984593


In [32]:
# Threshold for class prediction
chosen_threshold = 0.73# or test a few, as you did before
y_pred = (y_pred_prob_test >= chosen_threshold).astype(int)

# Evaluate
print("Confusion Matrix with SMOTE and threshold {:.2f}:".format(chosen_threshold))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1-Score: ", f1_score(y_test, y_pred))

Confusion Matrix with SMOTE and threshold 0.73:
[[2971  156]
 [ 246  326]]
              precision    recall  f1-score   support

           0       0.92      0.95      0.94      3127
           1       0.68      0.57      0.62       572

    accuracy                           0.89      3699
   macro avg       0.80      0.76      0.78      3699
weighted avg       0.89      0.89      0.89      3699

Precision: 0.6763485477178424
Recall:    0.5699300699300699
F1-Score:  0.618595825426945


In [40]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
param_grid = {
    'n_estimators':randint(100, 500),
    'max_features': ['sqrt', 'log2'],
    'max_depth':randint(5, 30),
    'min_samples_split':randint(2, 20),
    'min_samples_leaf':randint(1, 10),
    'class_weight': ['balanced', 'balanced_subsample', None]
}
rf = RandomForestClassifier(random_state=42)
search = RandomizedSearchCV(
    rf, param_grid, n_iter=50, scoring="f1", cv=5,verbose=1, n_jobs=-1, random_state=42
)
search.fit(X_train_scaled, y_train)
print("Best parameters:", search.best_params_)


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters: {'class_weight': 'balanced', 'max_depth': 18, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 9, 'n_estimators': 234}


In [41]:
best_params = {
    'n_estimators':234,
    'max_features': 'sqrt',
    'max_depth': 18,
    'min_samples_split':4,
    'min_samples_leaf':9,
    'class_weight': 'balanced'
}

# Initialize model with best parameters
final_model_f1 = RandomForestClassifier(
    **best_params,
    random_state=42,
)

# Train on the full training set
final_model_f1.fit(X_train_scaled, y_train)

# Predict
y_pred = final_model_f1.predict(X_test_scaled)
y_prob = final_model_f1.predict_proba(X_test_scaled)[:, 1]

# Evaluate
print("=== Final RF'f1 Performance ===")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

=== Final RF'f1 Performance ===
              precision    recall  f1-score   support

           0       0.95      0.89      0.92      3127
           1       0.56      0.75      0.64       572

    accuracy                           0.87      3699
   macro avg       0.76      0.82      0.78      3699
weighted avg       0.89      0.87      0.88      3699

Confusion Matrix:
[[2794  333]
 [ 142  430]]
ROC-AUC: 0.9216


In [42]:
best_params = {
    'n_estimators':330,
    'max_features': 'sqrt',
    'max_depth': 7,
    'min_samples_split':5,
    'min_samples_leaf':11,
    'class_weight': None
}

# Initialize model with best parameters
final_model = RandomForestClassifier(
    **best_params,
    random_state=42,
)

# Train on the full training set
final_model.fit(X_train_scaled, y_train)

# Predict
y_pred = final_model.predict(X_test_scaled)
y_prob = final_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate
print("=== Final RF'precision Performance ===")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

=== Final RF'precision Performance ===
              precision    recall  f1-score   support

           0       0.91      0.97      0.94      3127
           1       0.77      0.49      0.60       572

    accuracy                           0.90      3699
   macro avg       0.84      0.73      0.77      3699
weighted avg       0.89      0.90      0.89      3699

Confusion Matrix:
[[3042   85]
 [ 289  283]]
ROC-AUC: 0.9153
